# Geocoding de direcciones de CABA

## No ejecutar este notebook

El geocoding completo ya se corrió. Los resultados están guardados en la carpeta de salida y volver a correrlo lleva varias horas, además de pisar archivos que ya consume el resto del análisis.

Las celdas de código que están más abajo solamente leen los archivos de salida para mostrar los resultados. Si por alguna razón necesitás volver a correr el pipeline, hay que ejecutar el script desde la terminal.

## 1. Contexto

El dataset que tenemos limpio tiene alrededor de 52 mil avisos, pero la única información geográfica que viene en cada uno es la dirección como texto: calle, altura y barrio. Para los análisis espaciales necesitamos latitud y longitud por propiedad.

Lo que hicimos fue:

1. Identificar las direcciones únicas dentro del conjunto de avisos que tenían calle y altura, normalizando acentos, puntuación y abreviaturas como Av, Avda o Avenida.
2. Consultar cada dirección contra dos APIs distintas. Una del Gobierno de la Ciudad de Buenos Aires y otra del Estado Nacional.
3. Tomar como coordenada final el consenso entre las dos fuentes, calculando la mediana por eje.
4. Pegar el resultado de vuelta al dataframe completo.

## 2. Pipeline

El código completo vive en un script aparte. Acá va un resumen de cómo funciona:

1. Se carga el dataframe limpio. Se normaliza la dirección y se deduplica para quedarnos con las direcciones únicas.
2. Cada API tiene su propio cache en formato JSON. Cada respuesta cruda se guarda con su latitud, longitud, estado, etiqueta y timestamp. Esto sirve por si el script se interrumpe en el medio.
3. Hay un control de frecuencia por API con reintentos y backoff exponencial para errores transitorios. Además, un tope duro por intento usando un hilo aparte, porque el timeout del cliente HTTP no siempre llega al socket subyacente.
4. Para cada dirección resuelta por las dos APIs se calcula la distancia geodésica entre las dos coordenadas. Si están a pocos metros una de la otra, es señal de que el consenso es confiable.
5. El consenso es la mediana por eje entre las APIs que resolvieron esa dirección. Esto lo hace robusto a outliers de una sola fuente.
6. Para cada fila del dataframe completo, se busca por clave en la tabla de consenso y se traen las columnas de latitud, longitud y la fuente del geocoding.

## 3. Resultados finales

In [ ]:
from pathlib import Path

import pandas as pd

OUTDIR = Path('../data/geocoding')

df = pd.read_csv(OUTDIR / 'dataframe_con_coords.tsv', sep='\t', low_memory=False)
with_coords = df['lat'].notna()
print(f'Total de avisos:                       {len(df):>7,}')
print(f'Con calle y altura:                    {(df["calle"].notna() & df["altura"].notna()).sum():>7,}')
print(f'Con coordenadas resueltas:             {with_coords.sum():>7,}  ({100*with_coords.mean():.1f} %)')
print(f'Sin coordenadas (se excluyen):         {(~with_coords).sum():>5,}')

In [ ]:
# Cobertura por barrio: cuántos avisos quedaron con coordenadas
(df.groupby('barrio_oficial')
   .agg(total=('lat', 'size'), con_coords=('lat', 'count'))
   .assign(cobertura_pct=lambda x: (100 * x['con_coords'] / x['total']).round(1))
   .sort_values('cobertura_pct')
   .head(10))

### 3.1 Cruce entre las dos fuentes

Tomamos las direcciones que las dos APIs lograron resolver y miramos la distancia entre las dos coordenadas devueltas. Una distancia chica significa que las dos fuentes coincidieron en el punto y por lo tanto el consenso es confiable.

In [ ]:
sample = pd.read_csv(OUTDIR / 'geocoded_sample.tsv', sep='\t')
ambas = sample[sample['n_ok'] >= 2]
d = ambas['max_pairwise_m']

print(f'Direcciones resueltas por las dos APIs: {len(ambas):,} de un total de {len(sample):,}\n')
print('Distancia entre las dos fuentes sobre la misma dirección:')
print(f'  mediana                                {d.median():>7.1f} m')
print(f'  percentil 90                           {d.quantile(0.9):>7.1f} m')
print(f'  máxima                                 {d.max():>7.1f} m')
print()
print(f'  hasta 25 metros (mismo edificio)       {(d <=  25).sum():>7,} de {len(d):,}  ({100*(d <=  25).mean():.1f} %)')
print(f'  hasta 100 metros (misma cuadra)        {(d <= 100).sum():>7,} de {len(d):,}  ({100*(d <= 100).mean():.1f} %)')
print(f'  más de 500 metros (discrepan)          {(d >  500).sum():>7,} de {len(d):,}  ({100*(d >  500).mean():.1f} %)')

### 3.2 Outliers, direcciones donde las dos APIs no coinciden

En estos casos lo más probable es que las APIs hayan elegido calles distintas con el mismo nombre, algo común porque en la ciudad hay varias calles que repiten nombres como San Martín o Belgrano. El consenso, al ser la mediana entre las dos, termina cayendo en algún punto intermedio y resulta menos confiable. Para análisis sensibles a la precisión geográfica conviene filtrar estos casos o resolverlos a mano.

In [ ]:
outliers = sample[sample['max_pairwise_m'] > 500].sort_values('max_pairwise_m', ascending=False)
print(f'Cantidad de outliers con más de 500 metros de discrepancia: {len(outliers):,}\n')
outliers[['calle', 'altura', 'barrio', 'usig_lat', 'usig_lon', 'georef_lat', 'georef_lon', 'max_pairwise_m']].head(10)

## 4. Archivos producidos

El pipeline genera varios archivos. Por un lado, los caches con la respuesta cruda de cada API, que sirven como input para construir el consenso y permiten retomar el proceso si se interrumpe. Por otro lado, una tabla ancha con una fila por dirección única que tiene las coordenadas de cada API más las del consenso y la distancia máxima entre fuentes, que sirve para auditar los resultados y detectar outliers.

Después está el archivo principal, que es el dataframe limpio enriquecido con las columnas de latitud, longitud y la fuente del geocoding. Ese es el input directo del notebook de análisis. Finalmente, hay un reporte autogenerado con las métricas finales del pipeline, que queda como documentación.

## 5. Cómo se usa esto en el análisis

El notebook principal carga directamente el dataframe enriquecido y filtra las filas que no tienen coordenadas antes de calcular distancias o cualquier otro análisis espacial. Las filas sin coordenadas, que son alrededor del quince por ciento del total, se excluyen explícitamente para no introducir sesgo en los análisis geográficos. El resto del notebook las sigue usando para los análisis que no dependen de la ubicación.